In [1]:
import findspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [9]:
df = spark.read.csv('student_performance_large.csv',header=True)

In [10]:
df.take(3)

[Row(study_hours='2', attendance='41', assignments='5', sleep_hours='5', result='0'),
 Row(study_hours='4', attendance='48', assignments='2', sleep_hours='8', result='0'),
 Row(study_hours='2', attendance='77', assignments='7', sleep_hours='4', result='1')]

In [13]:
df.show()

+-----------+----------+-----------+-----------+------+
|study_hours|attendance|assignments|sleep_hours|result|
+-----------+----------+-----------+-----------+------+
|          2|        41|          5|          5|     0|
|          4|        48|          2|          8|     0|
|          2|        77|          7|          4|     1|
|          1|        45|          4|          5|     0|
|          9|        78|          1|          8|     1|
|          4|        85|          9|          7|     1|
|          4|        68|         10|          6|     1|
|          1|        88|          3|          7|     0|
|          6|        57|          3|          5|     1|
|          6|        46|          2|          7|     1|
|          2|        62|          6|          8|     1|
|          5|        91|          1|          7|     1|
|          9|        47|          7|          4|     1|
|          9|        58|         10|          6|     1|
|         10|        52|          2|          4|

In [16]:
df.printSchema()

root
 |-- study_hours: string (nullable = true)
 |-- attendance: string (nullable = true)
 |-- assignments: string (nullable = true)
 |-- sleep_hours: string (nullable = true)
 |-- result: string (nullable = true)



In [36]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType,IntegerType

In [38]:
for i in df.columns[0:4]:
    df = df.withColumn(i,col(i).cast(DoubleType()))

In [39]:
df = df.withColumn('result',col('result').cast(IntegerType()))

In [40]:
df.printSchema()

root
 |-- study_hours: double (nullable = true)
 |-- attendance: double (nullable = true)
 |-- assignments: double (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- result: integer (nullable = true)



In [41]:
from pyspark.sql.functions import count

Count the total number of students.

In [46]:
df.count()

150

Calculate the average study_hours for all students

In [52]:
from pyspark.sql.functions import avg
df.select(avg('study_hours')).show()

+----------------+
|avg(study_hours)|
+----------------+
|            5.18|
+----------------+



How many students studied more than 5 hours?

In [54]:
df.filter(col('study_hours')>5).count()

67

How many students have attendance less than 60?

In [55]:
df.filter(col('attendance')<60).count()

51

How many students passed and how many failed?


In [57]:
df.groupBy('result').count().show()

+------+-----+
|result|count|
+------+-----+
|     1|  126|
|     0|   24|
+------+-----+



In [87]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
vecassembler = VectorAssembler(inputCols=df.columns[0:4],outputCol='features')


In [88]:
df_assembled = vecassembler.transform(df)

In [89]:
df_assembled.printSchema()

root
 |-- study_hours: double (nullable = true)
 |-- attendance: double (nullable = true)
 |-- assignments: double (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- result: integer (nullable = true)
 |-- features: vector (nullable = true)



In [90]:
df1 = df_assembled.select(['features','result'])

In [91]:
df1.show()

+-------------------+------+
|           features|result|
+-------------------+------+
| [2.0,41.0,5.0,5.0]|     0|
| [4.0,48.0,2.0,8.0]|     0|
| [2.0,77.0,7.0,4.0]|     1|
| [1.0,45.0,4.0,5.0]|     0|
| [9.0,78.0,1.0,8.0]|     1|
| [4.0,85.0,9.0,7.0]|     1|
|[4.0,68.0,10.0,6.0]|     1|
| [1.0,88.0,3.0,7.0]|     0|
| [6.0,57.0,3.0,5.0]|     1|
| [6.0,46.0,2.0,7.0]|     1|
| [2.0,62.0,6.0,8.0]|     1|
| [5.0,91.0,1.0,7.0]|     1|
| [9.0,47.0,7.0,4.0]|     1|
|[9.0,58.0,10.0,6.0]|     1|
|[10.0,52.0,2.0,4.0]|     1|
| [4.0,89.0,5.0,4.0]|     1|
| [4.0,95.0,2.0,7.0]|     1|
| [5.0,69.0,6.0,5.0]|     1|
| [6.0,62.0,4.0,6.0]|     1|
| [2.0,78.0,3.0,8.0]|     0|
+-------------------+------+
only showing top 20 rows


In [92]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,withStd=True )

In [93]:
scaled_model = scaler.fit(df1)
scaled = scaled_model.transform(df1)

In [95]:
train,test = scaled.randomSplit([0.8,0.2],seed=42)

In [96]:
from pyspark.ml.classification import LogisticRegression
LR = LogisticRegression(featuresCol='scaled_features',labelCol='result',predictionCol='predicted_result')
fitted = LR.fit(train)

In [97]:
final_df = fitted.transform(test)
final_df.show()

+-------------------+------+--------------------+--------------------+--------------------+----------------+
|           features|result|     scaled_features|       rawPrediction|         probability|predicted_result|
+-------------------+------+--------------------+--------------------+--------------------+----------------+
| [1.0,45.0,4.0,5.0]|     0|[-1.4643167572801...|[154.063551337384...|           [1.0,0.0]|             0.0|
|[1.0,55.0,10.0,4.0]|     1|[-1.4643167572801...|[-194.06956875825...|[5.20783460348368...|             1.0|
| [1.0,77.0,8.0,8.0]|     1|[-1.4643167572801...|[-157.11491425221...|[5.83256672104653...|             1.0|
| [1.0,87.0,6.0,4.0]|     1|[-1.4643167572801...|[-54.913279345899...|[1.41731308566106...|             1.0|
| [2.0,43.0,7.0,6.0]|     1|[-1.1140017435767...|[-63.932525960139...|[1.71576087941251...|             1.0|
| [2.0,64.0,7.0,8.0]|     1|[-1.1140017435767...|[-124.88262490035...|[5.80985268036271...|             1.0|
| [2.0,77.0,7.0,4.0

In [98]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="result",
    predictionCol="predicted_result",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(final_df)
print(f"Accuracy: {accuracy}")

Accuracy: 1.0


In [99]:
train.show()

+-------------------+------+--------------------+
|           features|result|     scaled_features|
+-------------------+------+--------------------+
| [1.0,40.0,6.0,5.0]|     0|[-1.4643167572801...|
| [1.0,42.0,4.0,7.0]|     0|[-1.4643167572801...|
| [1.0,45.0,7.0,6.0]|     1|[-1.4643167572801...|
|[1.0,46.0,10.0,7.0]|     1|[-1.4643167572801...|
| [1.0,51.0,6.0,7.0]|     0|[-1.4643167572801...|
| [1.0,59.0,6.0,4.0]|     0|[-1.4643167572801...|
| [1.0,81.0,9.0,4.0]|     1|[-1.4643167572801...|
| [1.0,82.0,4.0,7.0]|     0|[-1.4643167572801...|
| [1.0,83.0,2.0,8.0]|     0|[-1.4643167572801...|
| [1.0,86.0,5.0,8.0]|     1|[-1.4643167572801...|
| [1.0,88.0,3.0,7.0]|     0|[-1.4643167572801...|
| [1.0,91.0,6.0,7.0]|     1|[-1.4643167572801...|
| [1.0,93.0,6.0,8.0]|     1|[-1.4643167572801...|
| [1.0,95.0,2.0,5.0]|     0|[-1.4643167572801...|
| [2.0,41.0,5.0,5.0]|     0|[-1.1140017435767...|
| [2.0,46.0,7.0,6.0]|     1|[-1.1140017435767...|
| [2.0,55.0,4.0,5.0]|     0|[-1.1140017435767...|
